[![Open In Colab](/_static/colab-badge.svg)](https://colab.research.google.com/github/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)
[![Get Notebook](/_static/get-notebook-badge.svg)](https://raw.githubusercontent.com/OpenProteinAI/openprotein-docs/refs/heads/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)
[![View In GitHub](/_static/view-in-github-badge.svg)](https://github.com/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-prediction/Using_ESMFold2.ipynb)

# Using ESMFold2

ESMFold2 is the latest generation of the ESM structure-prediction family
([Biohub/esm](https://github.com/Biohub/esm)). Unlike first-generation
ESMFold, it predicts full biomolecular *complexes* — multiple protein
chains, nucleic acids, and small-molecule ligands — with a
diffusion-based decoder, and can optionally condition on a multiple
sequence alignment (MSA) for improved accuracy.

Two variants are available:

- `` :py:class:`~openprotein.fold.ESMFold2Model` ``{=rst}
  (`session.fold.esmfold2`) — the full model; accepts an optional MSA
  per protein chain.
- `` :py:class:`~openprotein.fold.ESMFold2FastModel` ``{=rst}
  (`session.fold.esmfold2_fast`) — a single-sequence variant for fast
  predictions without an MSA.

> First-generation **ESMFold** is still available via
> `session.fold.esmfold` for quick single-chain, single-sequence
> predictions. It does *not* support ligands, nucleic acids, multi-chain
> complexes, or MSA conditioning — reach for ESMFold2 when you need any
> of those. See the note on ESMFold at the end of this guide.

## What you need before getting started

Connect to your session and define the sequence whose structure you want
to predict. The example used here is Interleukin-2:

In [ ]:
import openprotein
from openprotein.molecules import Complex, Ligand, Protein

# Login to your session
session = openprotein.connect()

sequence = "MYRMQLLSCIALSLALVTNSAPTSSSTKKTQLQLEHLLLDLQMILNGINNYKNPKLTRMLTFKFYMPKKATELKHLQCLEEELKPLEEVLNLAQSKNFHLRPRDLISNINVIVLELKGSEP"

## Getting the model

Create the model object for ESMFold2:

In [ ]:
esmfold2 = session.fold.esmfold2
help(esmfold2.fold)

## Predicting a structure

Every protein chain must declare how it is conditioned. For a quick
prediction without an MSA, tag the
`` :py:class:`~openprotein.molecules.Protein` ``{=rst} with
`Protein.single_sequence_mode`:

In [ ]:
chain = Protein(sequence)
chain.msa = Protein.single_sequence_mode

future = esmfold2.fold(
    [chain],
    num_recycles=3,        # trunk recycling iterations
    num_steps=200,         # diffusion sampling steps
    diffusion_samples=1,   # number of structure samples to draw
    seed=0,
)
future

The runtime hyperparameters trade speed for accuracy: `num_recycles`
controls how many times the trunk refines its representation,
`num_steps` is the number of diffusion sampling steps, and
`diffusion_samples` draws multiple independent structure samples per
input. `seed` makes a run reproducible.

You can fold a true multimer by passing a
`` :py:class:`~openprotein.molecules.Complex` ``{=rst} of named chains.
Each chain is conditioned independently:

In [ ]:
chain_a = Protein(sequence)
chain_a.msa = Protein.single_sequence_mode
chain_b = Protein(sequence)
chain_b.msa = Protein.single_sequence_mode

complex = Complex(chains={"A": chain_a, "B": chain_b})
future = esmfold2.fold([complex])

Wait for the job to complete with
`` :py:meth:`~openprotein.jobs.Future.wait_until_done` ``{=rst}:

In [ ]:
future.wait_until_done(verbose=True, timeout=600)

## Folding with a ligand

A key capability of ESMFold2 — unavailable in first-generation ESMFold —
is co-folding small-molecule ligands alongside protein chains. Add a
`` :py:class:`~openprotein.molecules.Ligand` ``{=rst} chain by SMILES
string or by Chemical Component Dictionary (CCD) code:

In [ ]:
complex_with_ligand = Complex(chains={"A": chain_a})

# By SMILES (e.g. ethanol):
complex_with_ligand.set_chain("L", Ligand(smiles="CCO"))

# ...or by CCD code (e.g. heme):
# complex_with_ligand.set_chain("L", Ligand(ccd="HEM"))

future = esmfold2.fold([complex_with_ligand])

## Conditioning on an MSA

The full `esmfold2` variant can condition on an MSA for improved
accuracy. Attach an MSA to a protein chain by assigning its `msa`
attribute an MSA created with
`` :py:meth:`session.align.create_msa <openprotein.align.AlignAPI.create_msa>` ``{=rst}:

In [ ]:
# msa = session.align.create_msa(sequence.encode())
# chain = Protein(sequence)
# chain.msa = msa  # an MSAFuture or MSA id
# future = esmfold2.fold([chain])

`esmfold2-fast` is a single-sequence model and rejects chains that carry
an MSA — use `Protein.single_sequence_mode` with it instead.

## Fast single-sequence predictions with ESMFold2-Fast

When you don't need an MSA, `esmfold2-fast` is a lighter-weight, faster
variant. It accepts the same inputs and hyperparameters, but every
protein chain must use `Protein.single_sequence_mode`:

In [ ]:
esmfold2_fast = session.fold.esmfold2_fast

chain = Protein(sequence)
chain.msa = Protein.single_sequence_mode
fast_future = esmfold2_fast.fold([chain])

## Retrieving the results

Fetch the results with
`` :py:meth:`~openprotein.fold.FoldResultFuture.get` ``{=rst}, which
returns a list of
`` :py:class:`~openprotein.molecules.Structure` ``{=rst} objects — one
per input. Each `Structure` holds one
`` :py:class:`~openprotein.molecules.Complex` ``{=rst} per diffusion
sample:

In [ ]:
results = future.get()
structure = results[0]
complex = structure[0]               # first diffusion sample
protein = complex.get_protein("A")   # chains are named alphabetically

print("Predicted structure:", structure)
print("Predicted protein sequence:", protein.sequence)

Visualize the structure using
[molviewspec](https://github.com/molstar/mol-view-spec):

In [ ]:
%pip install molviewspec
from molviewspec import create_builder

def display_structure(structure_string):
    builder = create_builder()
    structure = builder.download(url="mystructure.cif")\
        .parse(format="mmcif")\
        .model_structure()\
        .component()\
        .representation()\
        .color_from_source(schema="atom",
                            category_name="atom_site",
                            field_name="auth_asym_id",
                            palette={"kind": "categorical",  # color by chain
                                     "colors": ["blue", "red", "green", "orange"],
                                     "mode": "ordinal"}
                          )
    return builder.molstar_notebook(data={'mystructure.cif': structure_string}, width=500, height=400)

display_structure(structure.to_string(format="cif"))

### Confidence scores

ESMFold2 returns per-sample confidence scores via
`` :py:meth:`~openprotein.fold.FoldResultFuture.get_confidence` ``{=rst}.
Each entry is an
`` :py:class:`~openprotein.fold.ESMFold2Confidence` ``{=rst} with the
complex pTM/ipTM, the mean complex pLDDT, and per-chain breakdowns:

In [ ]:
confidence = future.get_confidence()[0]  # one list per input; one entry per diffusion sample
c = confidence[0]

print("pTM:", c.ptm)
print("ipTM:", c.iptm)
print("complex pLDDT:", c.complex_plddt)
print("per-chain pTM:", c.chains_ptm)
print("pairwise chain ipTM:", c.pair_chains_iptm)

### PAE and pLDDT arrays

The PAE (Predicted Aligned Error) is an N × N matrix estimating the
expected error between residue pairs; pLDDT is the per-residue
confidence. Both are returned as NumPy arrays, one per input:

In [ ]:
pae = future.get_pae()[0]
plddt = future.get_plddt()[0]

print("PAE matrix shape:", pae.shape)
print("pLDDT shape:", plddt.shape)

## A note on ESMFold (first generation)

First-generation **ESMFold** remains available via
`session.fold.esmfold` for quick single-chain, single-sequence structure
predictions:

In [ ]:
# esm = session.fold.esmfold.fold([sequence.encode()], num_recycles=1)
# structure = esm.get()[0]

ESMFold (v1) does *not* support ligands, nucleic acids, multi-chain
complexes, or MSA conditioning. Use ESMFold2 (or `esmfold2-fast`)
whenever you need any of those capabilities; otherwise the
first-generation model is a fast option for single-sequence monomers.

## Next steps

Save your structure for future use, or compare it against another
predictor such as [AlphaFold2](./Using_AlphaFold2.ipynb),
[Boltz](./Using_Boltz_1_and_Boltz_2.ipynb), or
[Protenix](./Using_Protenix.ipynb):

In [ ]:
with open("esmfold2_prediction.cif", "w") as f:
    f.write(structure.to_string(format="cif"))